# Battleship Tutorial
This tutorial demonstrates how to generate the data to train the transformer and how to use the inference code to play a game.

In [1]:
!git clone https://github.com/pjuangph/battleship.git

fatal: destination path 'battleship' already exists and is not an empty directory.


In [2]:
!cp battleship/*.py .
%load_ext autoreload
%autoreload 2

> Note: These tutorials are for you guys to play with the code. I do not use jupyter notebooks for any real development because a common trap is running the cells in the wrong order which can lead to massive confusion. It's good to use jupyter notebooks for demonstration purposes.

## Training
The training code generates the data for number of games given a specific ship size. Lets start with importing the headers.

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.cuda.empty_cache()
from battleship_train import generate_games, train
import pickle


In [4]:
board_height = 10
board_width = 10
SHIP_SIZES = [5, 4, 3, 3, 2]  # Example ship sizes
number_of_games = 100
# Generate training data
generate_games(number_of_games,board_height,board_width,SHIP_SIZES)
# Generate games creates a folder called data/
# Inside of the folder will have a pickle file containing the games

Generating Games to play


100%|██████████| 100/100 [00:00<00:00, 283.84it/s]


In [5]:
# Lets look at the contents
data = pickle.load(open('data/training_data.pickle','rb'))
print(data['src'].shape)
print(data['tgt'].shape)

(9000, 100)
(9000, 100)


If you are running the code as is, you'll notice the shape `src` and `tgt` to be 9000x100. This is because the code generates 1000 games however it also simulates 90 guesses for each game. There's a parameter in generate_games for how much of the board should be guessed by the human before the transformer starts to predict. The default value is 10%.

In [ ]:
train(resume_training=False,save_every_n_epoch=1,epochs=10) # This could take a while depending on the number of games and epochs and whether or not you're using a GPU

Train Loop


  0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Epoch: 0 Train Loss: 9.16e-01 Hits match 99.99 Matches 17.01:   8%|▊         | 4/50 [02:05<23:48, 31.06s/it]

In [ ]:
# These are the generated training data
data = pickle.load(open('data/training_data.pickle','wb'))
# Data is broken down into source and target
source = data['src']
target = data['tgt'] # Validation purposes

## Inference
Inference code is how the user interacts with the model. There are two inference codes written ai_helper and auto_game. Auto_game simulates the computer playing against itself where as ai_helper helps the user with the game.

In [ ]:
from battleship_inference import ai_helper, auto_game

Playing a single game will show the results in realtime so each guess, each move will be displayed. However if you were to run more tha one games then it will run without outputing the game.

In [ ]:
auto_game(n_games=1,train=False) # Run a game with the AI. Outputs the game board and the AI's moves.

In [ ]:
auto_game(n_games=10,train=False) # Run 10 games with the trained model but no outputs will be displayed

In [ ]:
ai_helper() # This will enable the AI to help you play a game of battleship against an opponent